In [37]:
!pip install torch nltk -q

In [38]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter
import math
import numpy as np
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [39]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [40]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [41]:
from pathlib import Path

def extract_text(path=Path("/content/drive/MyDrive/Deep_Learning/Coursework/61262-0.txt")):
    """
    Extract text from a .txt file using pathlib.Path.
    Default path points to Google Drive location.
    """

    # Ensure it's a Path object
    path = Path(path)

    # Check if file exists
    if not path.exists():
        raise FileNotFoundError(f"File not found at: {path}")

    # Ensure it's a .txt file
    if path.suffix.lower() != ".txt":
        raise ValueError("Provided file is not a .txt file")

    # Read text safely
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    return text


In [42]:
import re

def clean_text(text):
    """
    Light cleaning for book text (LM-friendly).

    - Removes Gutenberg license/header/footer
    - Removes standalone page numbers
    - Converts to lowercase
    - Keeps punctuation
    - Keeps chapter names
    - Normalizes whitespace
    """

    # --------------------------------------------------
    # 1. Remove Gutenberg Header
    # --------------------------------------------------
    start_pattern = r"\*\*\*\s*start of.*?\*\*\*"
    end_pattern = r"\*\*\*\s*end of.*?\*\*\*"

    start_match = re.search(start_pattern, text, re.IGNORECASE | re.DOTALL)
    end_match = re.search(end_pattern, text, re.IGNORECASE | re.DOTALL)

    if start_match:
        text = text[start_match.end():]

    if end_match:
        text = text[:end_match.start()]

    # --------------------------------------------------
    # 2. Remove standalone page numbers (e.g., "\n 23 \n")
    # --------------------------------------------------
    text = re.sub(r"\n\s*\d+\s*\n", "\n", text)

    # --------------------------------------------------
    # 3. Convert to lowercase
    # --------------------------------------------------
    text = text.lower()

    # --------------------------------------------------
    # 4. Normalize whitespace (keep punctuation)
    # --------------------------------------------------
    text = re.sub(r"\s+", " ", text)

    # --------------------------------------------------
    # 5. Strip edges
    # --------------------------------------------------
    text = text.strip()

    return text


In [43]:
import random
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize

def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ").lower()
            if synonym != word:
                synonyms.add(synonym)
    return list(synonyms)


def synonym_replacement(tokens, n=2):
    new_tokens = tokens.copy()
    words = list(set(tokens))
    random.shuffle(words)

    replaced = 0
    for word in words:
        synonyms = get_synonyms(word)
        if len(synonyms) > 0:
            synonym = random.choice(synonyms)
            new_tokens = [synonym if w == word else w for w in new_tokens]
            replaced += 1
        if replaced >= n:
            break

    return new_tokens


def random_deletion(tokens, p=0.05):
    if len(tokens) == 1:
        return tokens

    return [word for word in tokens if random.uniform(0,1) > p]


def random_swap(tokens, n=2):
    new_tokens = tokens.copy()
    for _ in range(n):
        idx1, idx2 = random.sample(range(len(new_tokens)), 2)
        new_tokens[idx1], new_tokens[idx2] = new_tokens[idx2], new_tokens[idx1]
    return new_tokens


def random_insertion(tokens, n=2):
    new_tokens = tokens.copy()

    for _ in range(n):
        word = random.choice(new_tokens)
        synonyms = get_synonyms(word)
        if synonyms:
            insert_word = random.choice(synonyms)
            insert_pos = random.randint(0, len(new_tokens))
            new_tokens.insert(insert_pos, insert_word)

    return new_tokens


def add_noise(tokens, p=0.03):
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    new_tokens = []

    for word in tokens:
        if random.uniform(0,1) < p:
            char_pos = random.randint(0, len(word)-1)
            random_char = random.choice(alphabet)
            word = word[:char_pos] + random_char + word[char_pos+1:]
        new_tokens.append(word)

    return new_tokens

def augment_text(text,
                 synonym_prob=0.3,
                 deletion_prob=0.05,
                 swap_prob=0.3,
                 insertion_prob=0.3,
                 noise_prob=0.02):

    # Clean
    text = text.lower()
    text = " ".join(text.split())

    tokens = text.split()

    # Apply augmentations conditionally
    if random.random() < synonym_prob:
        tokens = synonym_replacement(tokens, n=2)

    if random.random() < deletion_prob:
        tokens = random_deletion(tokens, p=0.05)

    if random.random() < swap_prob:
        tokens = random_swap(tokens, n=2)

    if random.random() < insertion_prob:
        tokens = random_insertion(tokens, n=2)

    tokens = add_noise(tokens, p=noise_prob)

    return " ".join(tokens)


In [44]:
def tokenize_text(text):
    tokens = text.split()
    return ['<bos>'] + tokens + ['<eos>']


In [45]:
from collections import Counter
import re

def build_vocab(tokens, min_freq=3, max_vocab_size=None, lowercase=True, normalize_numbers=True):
    """
    Improved vocabulary builder:
    - Sorts by frequency
    - Applies min_freq
    - Optional max_vocab_size
    - Adds LM-specific special tokens
    - Optionally lowercases and normalizes numbers
    - Breaks rare words into characters if freq < min_freq
    """

    if lowercase:
        tokens = [t.lower() for t in tokens]

    if normalize_numbers:
        tokens = ['<num>' if t.isdigit() else t for t in tokens]

    counter = Counter(tokens)

    # Sort by frequency (high → low)
    sorted_words = sorted(counter.items(), key=lambda x: x[1], reverse=True)

    filtered_words = []
    rare_subwords = set()

    for word, freq in sorted_words:
        if freq >= min_freq:
            filtered_words.append(word)
        else:
            # Break rare words into characters and add them to vocab
            for ch in word:
                rare_subwords.add(ch)

    # Apply max_vocab_size limit (only to frequent words)
    if max_vocab_size:
        filtered_words = filtered_words[:max_vocab_size]

    # Combine frequent words and rare subwords
    all_tokens = filtered_words + list(rare_subwords)

    # Special tokens
    special_tokens = ['<pad>', '<unk>', '<bos>', '<eos>', '<num>']

    # Remove duplicates while preserving order
    unique_tokens = []
    seen = set()

    for token in special_tokens + all_tokens:
        if token not in seen:
            unique_tokens.append(token)
            seen.add(token)

    vocab = unique_tokens

    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}

    print("Final vocab size:", len(word2idx))

    return word2idx, idx2word


In [46]:
def encode_tokens(tokens, word2idx):
    unk_idx = word2idx['<unk>']
    return [word2idx.get(t, unk_idx) for t in tokens]

In [47]:
def create_sequences(encoded_text, seq_length=30):
    n_samples = len(encoded_text) - seq_length

    inputs = []
    targets = []

    for i in range(n_samples):
        input_seq = encoded_text[i:i+seq_length]
        target = encoded_text[i+seq_length]

        inputs.append(torch.tensor(input_seq, dtype=torch.long))
        targets.append(torch.tensor(target, dtype=torch.long))

    return torch.stack(inputs), torch.stack(targets)


In [48]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_size=200, hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden=None):
        embedded = self.dropout(self.embedding(x))
        output, hidden = self.lstm(embedded, hidden)
        output = self.dropout(output)
        output = self.fc(output)  # predict all time steps
        return output, hidden


In [49]:
def train_model(model, inputs, targets, word2idx,
                epochs=(5, 15),
                batch_size=64,
                lr=0.001,
                device=None,
                patience=3,
                min_lr=1e-5):

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = torch.utils.data.TensorDataset(inputs, targets)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=1, min_lr=min_lr
    )

    vocab_size = len(word2idx)

    phases = [
        {'name': 'Phase 1: Freeze embeddings', 'epochs': epochs[0], 'freeze_embeddings': True},
        {'name': 'Phase 2: Fine-tuning', 'epochs': epochs[1], 'freeze_embeddings': False}
    ]

    for phase in phases:

        for param in model.embedding.parameters():
            param.requires_grad = not phase['freeze_embeddings']

        print(f"\n{phase['name']}")

        best_loss = float('inf')
        stop_counter = 0

        for epoch in range(phase['epochs']):

            total_loss = 0
            total_correct = 0
            total_samples = 0

            model.train()

            for x_batch, y_batch in loader:

                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)

                # SAFETY: ensure targets valid
                if torch.any(y_batch >= vocab_size) or torch.any(y_batch < 0):
                    raise ValueError("Target index out of vocabulary range!")

                optimizer.zero_grad()

                outputs, _ = model(x_batch)

                # Take last time step
                outputs = outputs[:, -1, :]

                loss = criterion(outputs, y_batch)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)
                optimizer.step()

                total_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                total_correct += (preds == y_batch).sum().item()
                total_samples += y_batch.size(0)

            avg_loss = total_loss / len(loader)
            try:
                perplexity = math.exp(avg_loss)
            except OverflowError:
                perplexity = float('inf')

            accuracy = total_correct / total_samples

            print(
                f"Epoch {epoch+1}/{phase['epochs']} | "
                f"Loss: {avg_loss:.4f} | "
                f"PPL: {perplexity:.4f} | "
                f"Acc: {accuracy:.4f}")

            scheduler.step(avg_loss)

            if avg_loss < best_loss:
                best_loss = avg_loss
                stop_counter = 0
                torch.save(model.state_dict(), "best_model.pt")
            else:
                stop_counter += 1
                if stop_counter >= patience:
                    print("Early stopping triggered")
                    model.load_state_dict(torch.load("best_model.pt"))
                    break


In [50]:
def generate_text(model, seed_text, word2idx, idx2word,
                  max_length=100, temperature=1.0, device=None):

    if device is None:
      device = next(model.parameters()).device
    model.eval()

    tokens = seed_text.lower().split()
    encoded = [word2idx.get(token, word2idx['<unk>']) for token in tokens]

    input_seq = torch.tensor(encoded[-30:], dtype=torch.long).unsqueeze(0).to(device)
    hidden = None

    generated_words = tokens.copy()

    with torch.no_grad():
        for _ in range(max_length):

            output, hidden = model(input_seq, hidden)

            # Take only last time step
            output = output[:, -1, :]   # shape: [1, vocab_size]

            # Apply temperature scaling correctly
            probabilities = torch.softmax(output / temperature, dim=-1)

            # Remove batch dimension
            probabilities = probabilities.squeeze(0)  # shape: [vocab_size]

            next_word_id = torch.multinomial(probabilities, 1).item()
            next_word = idx2word[next_word_id]

            generated_words.append(next_word)

            # Update input sequence (sliding window)
            input_seq = torch.cat(
                [input_seq[:, 1:], torch.tensor([[next_word_id]], device=device)],
                dim=1
            )

    return " ".join(generated_words)


In [51]:
def replace_unknowns_in_text(text, word2idx, idx2word):
    # Replace all <unk> tokens with a placeholder
    text = text.replace('<unk>', '[UNKNOWN]')

    if '<unk>' in text:
        text = text.replace('<unk>', 'sherrif')

    return text


In [52]:
import os
import torch
from datetime import datetime

def save_model_to_drive(model, word2idx, save_dir='/content/drive/MyDrive/Deep_Learning/Coursework/saved_models',
                        filename='lstm_model.pth'):
    """
    Saves:
    - model state_dict
    - vocabulary
    - timestamp
    Creates directory if it does not exist.
    """

    try:
        # Create directory if needed
        os.makedirs(save_dir, exist_ok=True)

        # Full file path
        save_path = os.path.join(save_dir, filename)

        # Package everything important
        checkpoint = {
            'model_state_dict': model.state_dict(),
            'vocab': word2idx,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        torch.save(checkpoint, save_path)

        print(f"Model saved successfully to:\n{save_path}")

    except Exception as e:
        print(f" Error saving model:\n{e}")


In [53]:
def load_model_from_drive(model_class, load_path, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    checkpoint = torch.load(load_path, map_location=device)

    vocab = checkpoint['vocab']
    model = model_class(len(vocab))
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    print("Model loaded successfully")
    return model, vocab

In [54]:
# 1. Load text
text = extract_text()

text = clean_text(text)

# 2. Augment
text = augment_text(text)

# 3. Tokenize
tokens = tokenize_text(text)

# 4. Build vocab
word2idx, idx2word = build_vocab(tokens)

# 5. Encode
encoded = encode_tokens(tokens, word2idx)

# 6. Create sequences (ensure targets are valid)
inputs, targets = create_sequences(encoded, seq_length=30)

# 7. Initialize model
model = LSTMLanguageModel(len(word2idx)).to(device)

print("Max target:", targets.max().item())
print("Vocab size:", len(word2idx))

# 8. Train (20 epochs total: 5 freeze + 15 unfreeze)
train_model(model, inputs, targets, word2idx,
            epochs=(5,15),
            batch_size=64,
            lr=0.001)

save_model_to_drive(model, word2idx)


# 9. Switch to evaluation mode

print("\nModel ready! Type a prompt (or 'quit' to exit):\n")

while True:
    user_input = input("Prompt: ").strip()

    if user_input.lower() in ["quit", "exit"]:
        print("Exiting...")
        break

    if len(user_input) == 0:
        continue

    output = generate_text(
        model,
        seed_text=user_input,
        word2idx=word2idx,
        idx2word=idx2word,
        max_length=120,
        temperature=0.8)

    output = replace_unknowns_in_text(output, word2idx, idx2word)

    print("\nGenerated Text:\n")
    print(output)
    print("-" * 50)


Final vocab size: 2390
Max target: 2389
Vocab size: 2390

Phase 1: Freeze embeddings
Epoch 1/5 | Loss: 5.4823 | PPL: 240.4011 | Acc: 0.1922
Epoch 2/5 | Loss: 5.1316 | PPL: 169.2815 | Acc: 0.1993
Epoch 3/5 | Loss: 4.9267 | PPL: 137.9247 | Acc: 0.2021
Epoch 4/5 | Loss: 4.7760 | PPL: 118.6249 | Acc: 0.2076
Epoch 5/5 | Loss: 4.6446 | PPL: 104.0189 | Acc: 0.2129

Phase 2: Fine-tuning
Epoch 1/15 | Loss: 4.5302 | PPL: 92.7747 | Acc: 0.2167
Epoch 2/15 | Loss: 4.4064 | PPL: 81.9719 | Acc: 0.2210
Epoch 3/15 | Loss: 4.2905 | PPL: 73.0001 | Acc: 0.2267
Epoch 4/15 | Loss: 4.1827 | PPL: 65.5422 | Acc: 0.2312
Epoch 5/15 | Loss: 4.0765 | PPL: 58.9384 | Acc: 0.2360
Epoch 6/15 | Loss: 3.9771 | PPL: 53.3625 | Acc: 0.2388
Epoch 7/15 | Loss: 3.8703 | PPL: 47.9572 | Acc: 0.2439
Epoch 8/15 | Loss: 3.7775 | PPL: 43.7073 | Acc: 0.2497
Epoch 9/15 | Loss: 3.6904 | PPL: 40.0607 | Acc: 0.2533
Epoch 10/15 | Loss: 3.6104 | PPL: 36.9790 | Acc: 0.2572
Epoch 11/15 | Loss: 3.5285 | PPL: 34.0716 | Acc: 0.2645
Epoch 12/15

In [55]:
save_model_to_drive(model, word2idx)

Model saved successfully to:
/content/drive/MyDrive/Deep_Learning/Coursework/saved_models/lstm_model.pth


In [56]:
#load model
load_path = "/content/drive/MyDrive/Deep_Learning/Coursework/saved_models/lstm_model.pth"

model, word2idx = load_model_from_drive(
    LSTMLanguageModel,
    load_path)

# rebuild idx2word
idx2word = {idx: word for word, idx in word2idx.items()}



Model loaded successfully


In [57]:
print("\nModel ready! Type a prompt (or 'quit' to exit):\n")

while True:
    user_input = input("Prompt: ").strip()

    if user_input.lower() in ["quit", "exit"]:
        print("Exiting...")
        break

    if len(user_input) == 0:
        continue

    output = generate_text(
        model,
        seed_text=user_input,
        word2idx=word2idx,
        idx2word=idx2word,
        max_length=120,
        temperature=0.8)

    output = replace_unknowns_in_text(output, word2idx, idx2word)

    print("\nGenerated Text:\n")
    print(output)
    print("-" * 50)


Model ready! Type a prompt (or 'quit' to exit):

Prompt: the man said

Generated Text:

the man said to [UNKNOWN] now on the next evening. i understand.” that he can have no [UNKNOWN] for one is [UNKNOWN] from the new [UNKNOWN] [UNKNOWN] a [UNKNOWN] [UNKNOWN] on the [UNKNOWN] [UNKNOWN] of the crime was [UNKNOWN] you must [UNKNOWN] the [UNKNOWN] he [UNKNOWN] poirot smiled into the [UNKNOWN] and [UNKNOWN] in on [UNKNOWN] [UNKNOWN] [UNKNOWN] [UNKNOWN] my idea was [UNKNOWN] by the [UNKNOWN] and [UNKNOWN] back and to the [UNKNOWN] of the [UNKNOWN] [UNKNOWN] out the order and the two [UNKNOWN] in [UNKNOWN] the [UNKNOWN] of the [UNKNOWN] of the italian [UNKNOWN] poirot shook his head. “but the great friend, literally to turn along the door was standing in the [UNKNOWN] i could be deeply [UNKNOWN] it seemed [UNKNOWN] to
--------------------------------------------------
Prompt: what?

Generated Text:

what? rest a [UNKNOWN] poirot was [UNKNOWN] between [UNKNOWN] beard, [UNKNOWN] [UNKNOWN] sai